# Feature Importances Analyse

* Tree Modelle Feature Importances
* Permutation Feature Importances Ranking

In [ ]:
# feature importances analysis of tree model
import sys
import os
import pickle
import importlib
import numpy as np
import pandas as pd

# Add the src directory to the system path to allow importing custom modules
sys.path.insert(0, os.path.abspath('../src'))

import warnings
warnings.filterwarnings('ignore')

# Enable autoreload to automatically reload modules when they are edited
%load_ext autoreload
%autoreload 2

# Set a consistent style for all plots
import matplotlib.pyplot as plt
plt.rcParams.update({
    'axes.grid':      True,
    'grid.color':     '#DCDCDC',
    'grid.linewidth': 0.5,
    'grid.linestyle': '-',
    'axes.axisbelow': True,
    'axes.facecolor': 'white',
    'font.family':    'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.titlepad':  13,
    'axes.labelsize': 10,
    'axes.labelpad':  8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'legend.frameon':    True,
    'legend.facecolor':  'white',
    'legend.edgecolor':  '#DCDCDC',
    'legend.framealpha': 1.0,
    'legend.fontsize':   9,
})

best_rf_model_bayesian = pickle.load(open('../models/best_rf_model_bayesian.pkl', 'rb'))

best_lgbm_model_bayesian = pickle.load(open('../models/best_lgbm_model_bayesian.pkl', 'rb'))

In [ ]:
# Conclusion: 
# Three models identify similar features as important 
#   - Energy load lag features and rolling statistics of the target variable are consistently ranked as top predictors across all three models
# This suggests that past values and trends in electricity load are key predictors for future demand.   


In [ ]:
# retrieve data and models
import pandas as pd
from fetch_prepare_data import *
from train_model_predict import *

df_for_modeling = pd.read_pickle('../data/processed/energy_weather_data_for_modeling.pkl')

features_train, target_train, features_test, target_test = train_test_split_by_date(df_for_modeling, 
                                                                                    date_column='time',
                                                                                    target_column='EnergyDemand', 
                                                                                    split_date='2025-01-01')
best_lgbm_model_bayesian = load_model_from_pickle('../models/best_lgbm_model_bayesian.pkl')
best_rf_model_bayesian = load_model_from_pickle('../models/best_rf_model_bayesian.pkl')
best_xgb_model_bayesian = load_model_from_pickle('../models/best_xgb_model_bayesian.pkl')

In [ ]:
# plot permutation feature importance of three selected models of top 20 features for MAE
from sklearn.inspection import permutation_importance

perm_importance_mae_lgbm = permutation_importance(best_lgbm_model_bayesian, features_test, target_test, scoring='neg_mean_absolute_error')
perm_importance_mae_xgb = permutation_importance(best_xgb_model_bayesian, features_test, target_test, scoring='neg_mean_absolute_error')
perm_importance_mae_rf = permutation_importance(best_rf_model_bayesian, features_test, target_test, scoring='neg_mean_absolute_error')
perm_importances_mae_lgbm = pd.Series(perm_importance_mae_lgbm.importances_mean, index=best_lgbm_model_bayesian.feature_names_in_).sort_values(ascending=True).tail(20)
perm_importances_mae_xgb = pd.Series(perm_importance_mae_xgb.importances_mean, index=best_xgb_model_bayesian.feature_names_in_).sort_values(ascending=True).tail(20)
perm_importances_mae_rf = pd.Series(perm_importance_mae_rf.importances_mean, index=best_rf_model_bayesian.feature_names_in_).sort_values(ascending=True).tail(20)

fig, axes = plt.subplots(1, 3, figsize=(17, 7))
perm_importances_mae_rf.plot.barh(ax=axes[0], color='steelblue') 
axes[0].set_title("Random Forest")
perm_importances_mae_lgbm.plot.barh(ax=axes[1], color='orange')
axes[1].set_title("LightGBM")
perm_importances_mae_xgb.plot.barh(ax=axes[2], color='mediumseagreen')
axes[2].set_title("XGBoost")
plt.tight_layout()
plt.show()
